<a href="https://colab.research.google.com/github/Janet-Wanjiru/my-fourth-ML-project-predict_health_costs_with_regression/blob/main/fcc_predict_health_costs_with_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Import libraries. You may or may not use all of these.
!pip install -q git+https://github.com/tensorflow/docs
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
  # %tensorflow_version only exists in Colab.
  %tensorflow_version 2.x
except Exception:
  pass
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers, models

import tensorflow_docs as tfdocs
import tensorflow_docs.plots
import tensorflow_docs.modeling

In [ ]:
# Import data
!wget https://cdn.freecodecamp.org/project-data/health-costs/insurance.csv
dataset = pd.read_csv('insurance.csv')
dataset.tail() #rprints out the last n rows, default 5

In [ ]:
categorical_cols = ['sex', 'smoker', 'region'] #Convert categorical string text columns into numerical indicator columns (One-Hot Encoding)
dataset = pd.get_dummies(dataset, columns=categorical_cols, drop_first=True)

print(dataset.tail()) #visualize all converted text

In [ ]:
train_dataset = dataset.sample(frac=0.8, random_state=42) #.sample shuffles and selects rows randomly, frac=0 gives percentage to build training dataset
test_dataset = dataset.drop(train_dataset.index) #drops all rows selected above leaving remaining 20% as testing data

train_labels = train_dataset.pop('expenses') #remove expenses as it is the thing we are tryinbg to predict
test_labels = test_dataset.pop('expenses')


In [ ]:

model = models.Sequential([ #creates a linear stacked file of layers where output flows from one layer to the next
    layers.Dense(64, activation='relu', input_shape=[len(train_dataset.keys())]), #first hidden layer with 64 neurons, normalize with relu for 0 to infinity, input shape tells how many numeric values in training dataset
    layers.Dense(64, activation='relu'),
    layers.Dense(32, activation='relu'),
    layers.Dense(1) # Exact single continuous regression output layer
])

model.compile(
    optimizer=tf.keras.optimizers.RMSprop(learning_rate=0.01), #sets optimizer that updates model's weights according to errors
    loss='mse',      # Mean Squared Error for training updates
    metrics=['mae', 'mse'] # Mean Absolute Error tracking for the grader challenge
)


In [ ]:
#model training
history = model.fit( #training g
    train_dataset,
    train_labels,
    epochs=300,
    validation_split=0.2, #grabs 20% of training data to use as validation during training
    verbose=0 # Suppresses massive text log tables
)

print("Training finished! Ready for final evaluation check.")


In [ ]:
# RUN THIS CELL TO TEST YOUR MODEL. DO NOT MODIFY CONTENTS.
# Test model by checking how well the model generalizes using the test set.
loss, mae, mse = model.evaluate(test_dataset, test_labels, verbose=2)

print("Testing set Mean Abs Error: {:5.2f} expenses".format(mae))

if mae < 3500:
  print("You passed the challenge. Great job!")
else:
  print("The Mean Abs Error must be less than 3500. Keep trying.")

# Plot predictions.
test_predictions = model.predict(test_dataset).flatten()

a = plt.axes(aspect='equal')
plt.scatter(test_labels, test_predictions)
plt.xlabel('True values (expenses)')
plt.ylabel('Predictions (expenses)')
lims = [0, 50000]
plt.xlim(lims)
plt.ylim(lims)
_ = plt.plot(lims,lims)
